Mp2 results

In [1]:
import pandas as pd
from mandelprot_lecture_4 import benchmark_mandelbrot_all

xmin, xmax = -2.0, 1.0
ymin, ymax = -1.5, 1.5
width, height = 2000, 1500
max_iter = 500
chunk_rows = 64

baseline, rows = benchmark_mandelbrot_all(
    xmin,
    xmax,
    ymin,
    ymax,
    width,
    height,
    max_iter,
    chunk_rows=chunk_rows,
    process_counts=[1, 2, 4, 8],
    runs=5,
)

df = pd.DataFrame(rows)
print("Sequential baseline:", baseline)
df

Sequential baseline: 0.5809248000005027


,processes,median_time_s,min_time_s,max_time_s,speedup,efficiency
0,1,1.595411,1.552589,1.718707,0.364122,0.364122
1,2,1.245239,1.228895,1.251493,0.466517,0.233258
2,4,1.151602,1.134283,1.183512,0.504449,0.126112
3,8,1.332930,1.291599,1.346648,0.435825,0.054478


The results show that the parallel implementation was slower than the , since the speedup remained below 1.0 in every case.

A explanation could be that the sequential version was already highly optimized by Numba, so the remaining work was not large enough to offset the overhead introduced by multiprocessing.

This is consistent with the general idea behind Amdahl’s Law, where the achievable benefit from parallelism is limited by serial work and parallel overhead.

Lecture 5 results

In [ ]:
from IPython.display import Image, display
from mandelprot_lecture_5 import (
    plot_chunk_time,
    plot_speedup,
    run_lecture5_study,
    to_dataframes,
)

results = run_lecture5_study(
    width=2000,
    height=1500,
    max_iter=500,
    n_workers_l04=4,
    process_counts=(1, 2, 4, 8),
    runs=3,
)

chunk_df, speed_df, compare_df = to_dataframes(results)

print("M1 same image:", results["m1_same_image"])
print("M1 chunks =", results["m1_chunks_default"], "time:", round(results["m1_time_default"], 4), "s")
print("M1 chunks =", results["m1_chunks_4x"], "time:", round(results["m1_time_4x"], 4), "s")
print("M2 best chunk row:", results["best_chunk_row"])
print("Estimated serial fraction s:", round(results["serial_fraction"], 6))
print("Tracker (1024x1024) median:", round(results["tracker_1024"]["median_s"], 4), "s")

display(chunk_df)
display(speed_df)
display(compare_df)

chunk_plot_path = plot_chunk_time(results["chunk_rows"], "lecture5_chunk_time.png")
speed_plot_path = plot_speedup(results["speed_rows"], "lecture5_speedup.png")

display(Image(filename=chunk_plot_path))
display(Image(filename=speed_plot_path))


In [ ]:
# Short interpretation generated from current run
best_speed_row = max(results["speed_rows"], key=lambda r: r["speedup"])
best_chunk_row = results["best_chunk_row"]

print("Interpretation")
print("- Fastest chunk setup:", best_chunk_row["n_chunks"], "chunks (LIF =", round(best_chunk_row["LIF"], 4), ")")
print("- Best speedup point:", best_speed_row["processes"], "processes with S(p)=", round(best_speed_row["speedup"], 4))
print("- Multiprocessing worth it here:", best_speed_row["speedup"] > 1.0)
print("- Why chunking helps: equal chunk sizes can still have unequal work in Mandelbrot rows.")
print("- Granularity tradeoff: too few chunks => load imbalance, too many => overhead.")
print("- Mandelbrot is embarrassingly parallel because pixels/chunks are independent.")
